# NBA 3.1 - 11 Adoption features (per claims cohort)

Builds the adoption features in `Feature Analysis/NBA_3_1_Adoption_Features_Inv_Far_Recommendations.md` (section 1) for every 3.1 universe HCP and claims cohort.

| # | Feature | Definition |
| --- | --- | --- |
| 1 | `MIN_DAYS_TO_PRESCRIBE_{DRUG}_POST_CKD_DIAGNOSIS` | NBA 3.0 `TIME_TO_ADOPT_{DRUG}`, renamed. Min over the HCP's patients of days from the patient's first CKD Dx **by this HCP** to the patient's first Rx of the drug from this HCP (negatives dropped) |
| 2 | `HCP_TIME_TO_ADOPT_{DRUG}` | Days from the drug's FDA approval to the HCP's first-ever Rx of the drug (any patient). Not built for `STEROIDAL_MRA` |
| 3 | `MIN_DAYS_TO_PRESCRIBE_{DRUG}_POST_FIRST_CKD_DIAGNOSIS_ANY_HCP` | Same as #1, but from the patient's first CKD Dx by **any** provider |

| `{DRUG}` | `PRODUCT_NAME` in `Rx_01_new` |
| --- | --- |
| `INVOKANA` | `INVOKANA` |
| `FARXIGA` | `FARXIGA` |
| `JARDIANCE` | `JARDIANCE` |
| `STEROIDAL_MRA` | `SPIRONOLACTONE`, `EPLERENONE` |
| `ENTRESTO` | `ENTRESTO` |

14 feature columns. Each cohort uses claims up to the end of its `lookback_end_date` month (no leakage). All claim types, as in NBA 3.0. Empty (null) when the HCP has no qualifying patient / Rx by the cutoff, as in NBA 3.0.

Inputs: 3.1 `Dx_03` and `Rx_01_new` (re-pulled with the 5 drug groups by `00_Write_HCP_Inputs_31`). Output: `kerendia_nba_3_1_volume/notebook_output/hcp_features/adoption_features`, keyed on `HCP_COHORT_ID`. No Snowflake needed (runs on serverless).

In [0]:
import os, json, builtins
from datetime import date
from datetime import datetime as dt
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.window import Window

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
COHORTS_DIR = f"{VOL31}/cohorts/claims"
HCP_UNIVERSE_PATH = f"{VOL31}/HCP_Universe"
QUERY_OUTPUT_31 = f"{VOL31}/hcp_level_model/ops_pipeline/query_output"
HCP_FEATURES_OUT_31 = f"{VOL31}/notebook_output/hcp_features"
OUTPUT_PATH = f"{HCP_FEATURES_OUT_31}/adoption_features"

# Drug groups -> PRODUCT_NAME values in Rx_01_new
DRUG_GROUPS = {
    "INVOKANA":      ["INVOKANA"],
    "FARXIGA":       ["FARXIGA"],
    "JARDIANCE":     ["JARDIANCE"],
    "STEROIDAL_MRA": ["SPIRONOLACTONE", "EPLERENONE"],
    "ENTRESTO":      ["ENTRESTO"],
}
# FDA approval dates for #2 (confirm with Medical). No entry = #2 not built (steroidal MRA has no launch date)
APPROVAL_DATES = {
    "INVOKANA":  "2013-03-29",
    "FARXIGA":   "2014-01-08",
    "JARDIANCE": "2014-08-01",
    "ENTRESTO":  "2015-07-07",
}
# NBA 3.0 date floors for #1 / #3 (Dx and Rx on or after the CKD-indication approval, as in 11_adoption_features_inv_far).
# Invokana's floor (2019-10-01) is before the data start (2021-01-02), so it has no effect.
DATE_FLOORS = {"INVOKANA": "2019-10-01", "FARXIGA": "2021-04-01"}

def f1(g): return f"MIN_DAYS_TO_PRESCRIBE_{g}_POST_CKD_DIAGNOSIS"
def f2(g): return f"HCP_TIME_TO_ADOPT_{g}"
def f3(g): return f"MIN_DAYS_TO_PRESCRIBE_{g}_POST_FIRST_CKD_DIAGNOSIS_ANY_HCP"

FEATURE_COLS = []
for g in DRUG_GROUPS:
    FEATURE_COLS.append(f1(g))
    if g in APPROVAL_DATES:
        FEATURE_COLS.append(f2(g))
    FEATURE_COLS.append(f3(g))
print(len(FEATURE_COLS), "features")
assert len(FEATURE_COLS) == 14

In [0]:
cohorts = pd.read_parquet(f"{COHORTS_DIR}/cohorts.parquet")
cohorts = cohorts.sort_values("cohort_id", key=lambda s: s.str[1:].astype(int)).reset_index(drop=True)
with open(f"{COHORTS_DIR}/metadata.json") as f:
    meta = json.load(f)
assert meta["feature_family"] == "claims", meta["feature_family"]
assert len(cohorts) == meta["cohort_count"] and (cohorts["offset_days"] == 0).all()

# Cutoff month = lookback_end_date (first of month); claims up to the end of that month are used
coh = (spark.createDataFrame(cohorts[["cohort_id", "lookback_end_date"]].astype(str))
       .withColumn("LOOKBACK_END_M", F.to_date("lookback_end_date")).drop("lookback_end_date"))

hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
n_hcp = hcp_universe.count()
print(f"Claims cohorts: {len(cohorts)} ({cohorts.cohort_id.iloc[0]}..{cohorts.cohort_id.iloc[-1]}), "
      f"lookback end {cohorts.lookback_end_date.iloc[0]}..{cohorts.lookback_end_date.iloc[-1]} | HCP universe: {n_hcp:,}")

In [0]:
dx = (spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Dx_03")
      .select("PROVIDER_ID", "PATIENT_ID", F.to_date("SVC_DT").alias("SVC_DT"))
      .filter(F.col("PROVIDER_ID").isNotNull() & F.col("PATIENT_ID").isNotNull()))

prod_to_grp = F.create_map(*[x for g, ps in DRUG_GROUPS.items() for p in ps for x in (F.lit(p), F.lit(g))])
all_products = [p for ps in DRUG_GROUPS.values() for p in ps]
rx = (spark.read.format("delta").load(f"{QUERY_OUTPUT_31}/Rx_01_new")
      .filter(F.col("PRODUCT_NAME").isin(all_products))
      .select("PROVIDER_ID", "PATIENT_ID", F.to_date("SVC_DT").alias("SVC_DT"), prod_to_grp[F.col("PRODUCT_NAME")].alias("GRP"))
      .filter(F.col("PROVIDER_ID").isNotNull() & F.col("PATIENT_ID").isNotNull()))

# Every drug group must be in Rx_01_new (it held only Invokana / Farxiga before the re-pull)
rx_summary = rx.groupBy("GRP").agg(F.count("*").alias("rx_rows"), F.countDistinct("PROVIDER_ID").alias("hcps"),
                                   F.min("SVC_DT").alias("first_svc_dt"), F.max("SVC_DT").alias("last_svc_dt")).orderBy("GRP")
display(rx_summary)
missing = set(DRUG_GROUPS) - {r["GRP"] for r in rx_summary.collect()}
assert not missing, f"Drug groups missing from Rx_01_new - re-pull with 00_Write_HCP_Inputs_31: {missing}"

In [0]:
# First dates over the full history. For a cohort with cutoff C, a patient counts when the first Rx is <= C
# (gap >= 0 means the first CKD date is also <= C), which gives the same values as recomputing with SVC_DT <= C.
floor_col = F.create_map(*[x for g, d in DATE_FLOORS.items() for x in (F.lit(g), F.lit(d).cast("date"))])

# Rx: first Rx per HCP-patient-group (on/after the group's 3.0 date floor)
rx_pair = (rx.withColumn("FLOOR", floor_col[F.col("GRP")])
             .filter(F.col("FLOOR").isNull() | (F.col("SVC_DT") >= F.col("FLOOR")))
             .groupBy("PROVIDER_ID", "PATIENT_ID", "GRP").agg(F.min("SVC_DT").alias("RX_DT")))

# Dx: first CKD date per HCP-patient (#1) and per patient from any provider (#3), for each distinct date floor
floors = sorted({DATE_FLOORS.get(g) for g in DRUG_GROUPS}, key=lambda x: x or "")
gap_parts = []
for fl in floors:
    grps = [g for g in DRUG_GROUPS if DATE_FLOORS.get(g) == fl]
    dx_f = dx if fl is None else dx.filter(F.col("SVC_DT") >= F.lit(fl).cast("date"))
    dx_pair = dx_f.groupBy("PROVIDER_ID", "PATIENT_ID").agg(F.min("SVC_DT").alias("CKD_DT"))
    dx_pat = dx_f.groupBy("PATIENT_ID").agg(F.min("SVC_DT").alias("CKD_DT_ANY"))
    gap_parts.append(rx_pair.filter(F.col("GRP").isin(grps))
                     .join(dx_pair, ["PROVIDER_ID", "PATIENT_ID"], "left")
                     .join(dx_pat, "PATIENT_ID", "left")
                     .withColumn("GAP_SAME", F.datediff("RX_DT", "CKD_DT"))
                     .withColumn("GAP_ANY", F.datediff("RX_DT", "CKD_DT_ANY")))
gaps = gap_parts[0]
for p in gap_parts[1:]:
    gaps = gaps.unionByName(p)

# Smallest non-negative gap per HCP, group and Rx month (cohort cutoffs are month ends)
gaps_m = (gaps.withColumn("RX_M", F.trunc("RX_DT", "month"))
          .groupBy("PROVIDER_ID", "GRP", "RX_M")
          .agg(F.min(F.when(F.col("GAP_SAME") >= 0, F.col("GAP_SAME"))).alias("GAP_SAME"),
               F.min(F.when(F.col("GAP_ANY") >= 0, F.col("GAP_ANY"))).alias("GAP_ANY"))
          .filter(F.col("GAP_SAME").isNotNull() | F.col("GAP_ANY").isNotNull()))

# #2: HCP's first-ever Rx per group (any patient, no date floor)
first_rx = rx.groupBy("PROVIDER_ID", "GRP").agg(F.min("SVC_DT").alias("FIRST_RX_DT"))

In [0]:
# #1 and #3: min gap over Rx months up to the cohort's lookback-end month
min_gaps = (gaps_m.join(F.broadcast(coh), F.col("RX_M") <= F.col("LOOKBACK_END_M"))
            .groupBy("PROVIDER_ID", "cohort_id", "GRP")
            .agg(F.min("GAP_SAME").alias("F1"), F.min("GAP_ANY").alias("F3")))

# #2: days from approval to first Rx, if the first Rx is on or before the cutoff (month end)
appr_col = F.create_map(*[x for g, d in APPROVAL_DATES.items() for x in (F.lit(g), F.lit(d).cast("date"))])
adopt = (first_rx.filter(F.col("GRP").isin(list(APPROVAL_DATES)))
         .join(F.broadcast(coh), F.col("FIRST_RX_DT") <= F.last_day("LOOKBACK_END_M"))
         .select("PROVIDER_ID", "cohort_id", "GRP", F.datediff("FIRST_RX_DT", appr_col[F.col("GRP")]).alias("F2")))

long = (min_gaps.join(adopt, ["PROVIDER_ID", "cohort_id", "GRP"], "full"))
aggs = []
for g in DRUG_GROUPS:
    aggs.append(F.min(F.when(F.col("GRP") == g, F.col("F1"))).alias(f1(g)))
    if g in APPROVAL_DATES:
        aggs.append(F.min(F.when(F.col("GRP") == g, F.col("F2"))).alias(f2(g)))
    aggs.append(F.min(F.when(F.col("GRP") == g, F.col("F3"))).alias(f3(g)))
wide = long.groupBy("PROVIDER_ID", "cohort_id").agg(*aggs)

# Every universe HCP x every claims cohort; nulls stay null (as in NBA 3.0)
adoption_features = (hcp_universe.crossJoin(F.broadcast(coh))
    .join(wide, ["PROVIDER_ID", "cohort_id"], "left")
    .withColumnRenamed("LOOKBACK_END_M", "SVC_DT_M")
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "cohort_id"))
    .withColumn("PULL_DT", F.lit(dt.today().strftime("%Y_%m_%d")))
    .select("PROVIDER_ID", "cohort_id", "SVC_DT_M", "HCP_COHORT_ID",
            *[F.col(c).cast("int").alias(c) for c in FEATURE_COLS], "PULL_DT"))

os.makedirs(HCP_FEATURES_OUT_31, exist_ok=True)
(adoption_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(OUTPUT_PATH))
afe = spark.read.format("delta").load(OUTPUT_PATH)
print(f"Saved {afe.count():,} rows, {len(afe.columns)} columns -> {OUTPUT_PATH}")

In [0]:
n_rows = afe.count()
assert n_rows == n_hcp * len(cohorts), (n_rows, n_hcp * len(cohorts))
assert afe.select("HCP_COHORT_ID").distinct().count() == n_rows, "HCP_COHORT_ID not unique"
assert afe.join(hcp_universe, "PROVIDER_ID", "left_anti").count() == 0
min_cols = [c for c in FEATURE_COLS if c.startswith("MIN_DAYS")]
neg = afe.filter(" OR ".join(f"{c} < 0" for c in min_cols)).count()

# Later cohorts see a superset of claims: a min can only fall, a value never disappears, #2 never changes once set
wc = Window.partitionBy("PROVIDER_ID").orderBy("SVC_DT_M")
prev = afe.select("PROVIDER_ID", "SVC_DT_M", *[F.lag(c).over(wc).alias(f"p_{c}") for c in FEATURE_COLS], *FEATURE_COLS)
bad_min = prev.filter(" OR ".join(f"(p_{c} IS NOT NULL AND ({c} IS NULL OR {c} > p_{c}))" for c in min_cols)).count()
bad_adopt = prev.filter(" OR ".join(f"(p_{c} IS NOT NULL AND ({c} IS NULL OR {c} != p_{c}))"
                                    for c in FEATURE_COLS if c.startswith("HCP_TIME"))).count()
print(f"rows {n_rows:,} | negative mins {neg} | min rose or vanished in a later cohort {bad_min} | #2 changed {bad_adopt}")
assert neg == 0 and bad_min == 0 and bad_adopt == 0

# Independent check: recompute the first and last cohort directly from claims with SVC_DT <= cutoff (NBA 3.0 style)
def recompute(cutoff_month):
    cut = F.last_day(F.lit(cutoff_month).cast("date"))
    dxc, rxc = dx.filter(F.col("SVC_DT") <= cut), rx.filter(F.col("SVC_DT") <= cut)
    cols = []
    for g, prods in DRUG_GROUPS.items():
        fl = DATE_FLOORS.get(g)
        dxg = dxc if fl is None else dxc.filter(F.col("SVC_DT") >= F.lit(fl).cast("date"))
        rxg = rxc.filter(F.col("GRP") == g)
        rxg_f = rxg if fl is None else rxg.filter(F.col("SVC_DT") >= F.lit(fl).cast("date"))
        pr = rxg_f.groupBy("PROVIDER_ID", "PATIENT_ID").agg(F.min("SVC_DT").alias("R"))
        same = (pr.join(dxg.groupBy("PROVIDER_ID", "PATIENT_ID").agg(F.min("SVC_DT").alias("D")), ["PROVIDER_ID", "PATIENT_ID"])
                  .withColumn("G", F.datediff("R", "D")).filter("G >= 0").groupBy("PROVIDER_ID").agg(F.min("G").alias(f1(g))))
        anyp = (pr.join(dxg.groupBy("PATIENT_ID").agg(F.min("SVC_DT").alias("D")), "PATIENT_ID")
                  .withColumn("G", F.datediff("R", "D")).filter("G >= 0").groupBy("PROVIDER_ID").agg(F.min("G").alias(f3(g))))
        cols += [same, anyp]
        if g in APPROVAL_DATES:
            cols.append(rxg.groupBy("PROVIDER_ID").agg(F.datediff(F.min("SVC_DT"), F.lit(APPROVAL_DATES[g]).cast("date")).alias(f2(g))))
    out = hcp_universe
    for c in cols:
        out = out.join(c, "PROVIDER_ID", "left")
    return out

for _, c in cohorts.iloc[[0, -1]].iterrows():
    chk = recompute(c.lookback_end_date).select("PROVIDER_ID", *[F.col(x).alias(f"chk_{x}") for x in FEATURE_COLS])
    cmp = afe.filter(F.col("cohort_id") == c.cohort_id).join(chk, "PROVIDER_ID")
    mism = cmp.filter(" OR ".join(f"NOT ({x} <=> chk_{x})" for x in FEATURE_COLS)).count()
    print(f"{c.cohort_id} (cutoff {c.lookback_end_date}): mismatches vs direct recompute = {mism}")
    assert mism == 0

In [0]:
last = cohorts.cohort_id.iloc[-1]
last_df = afe.filter(F.col("cohort_id") == last)
stats = last_df.agg(*[F.count(c).alias(f"n_{c}") for c in FEATURE_COLS],
                    *[F.percentile_approx(c, 0.5).alias(f"med_{c}") for c in FEATURE_COLS]).collect()[0]
cov = pd.DataFrame({"feature": FEATURE_COLS,
                    "hcps_with_value": [stats[f"n_{c}"] for c in FEATURE_COLS],
                    "pct_universe": [builtins.round(100 * stats[f"n_{c}"] / n_hcp, 1) for c in FEATURE_COLS],
                    "median_days": [stats[f"med_{c}"] for c in FEATURE_COLS]})
print(f"Cohort {last}")
display(cov)